# GLM-5.3-Flash on Amazon SageMaker AI, end to end

Deploy `zai-org/GLM-5.3-Flash` (321.3 B total / 18 B active, natively FP8) to a SageMaker AI
real-time endpoint on a custom vLLM container, smoke test it, benchmark it, read GPU
telemetry back, derive cost per token, and tear it down.

Written from an actual bring-up on `ml.p5en.48xlarge` in `us-east-2`. Every number quoted as
*measured* came off that run. Constraints that are not obvious from the docs are called out
inline rather than left to be discovered at 30 minutes per attempt.

## The five stages

| stage | one-time? | wall clock |
| --- | --- | --- |
| 1. Build the container | yes | ~10 min (CodeBuild) |
| 2. Stage weights to S3 | yes | ~15 min (measured 305.8 GiB at 353 MiB/s) |
| 3. Create model + endpoint config + endpoint | per deploy | ~15 min if capacity exists |
| 4. Smoke test and benchmark | per deploy | ~25 min for a 5-point sweep |
| 5. Delete | per deploy | ~5 min |

## Prerequisites

- An AWS account with SageMaker AI access and quota for an 8-GPU instance type.
- An execution role carrying `AmazonSageMakerFullAccess` (see the IAM naming trap below).
- A Hugging Face token with access to the model repo.
- `boto3`. The SageMaker Python SDK is only needed for the optional benchmark path in stage 4.

## Read this before you run anything

**Three resource names are load-bearing.** The managed IAM policies are prefix-scoped, so
naming things correctly avoids needing any permission changes at all:

| resource | must be named | why |
| --- | --- | --- |
| ECR repository | `sagemaker-*` | the CodeBuild service role policy allows ECR push only on `repository/sagemaker-*` |
| S3 staging bucket | contains `sagemaker` | `AmazonSageMakerFullAccess` scopes S3 object access to `*sagemaker*` |
| Secrets Manager secret | `AmazonSageMaker-*` | that policy scopes `GetSecretValue` to `secret:AmazonSageMaker-*` |

**Capacity, not configuration, is the hard part.** `InsufficientInstanceCapacity` takes about
31 minutes to surface and was the single largest time sink in this project. Failed attempts
provision nothing and cost nothing, but they cost you half an hour each. Stage 3 shows the two
mitigations.

---
## 0. Session and configuration

Edit this cell and nothing else. Everything downstream reads from it.

In [ ]:
import json
import time
import boto3
import botocore

PROFILE = None                    # None uses the default credential chain;
                                  # set to a named profile string if you use one
REGION = "us-east-2"

HF_MODEL_ID = "zai-org/GLM-5.3-Flash"
SERVED_MODEL_NAME = "glm-5.3-flash"

ECR_REPO = "sagemaker-vllm"       # must start with 'sagemaker-' (see IAM note above)
IMAGE_TAG = "glm53-flash"

INSTANCE_TYPE = "ml.p5en.48xlarge"

# Ordered fallback, highest priority first. Set to None to pin INSTANCE_TYPE instead.
# Caveat: pools give you ONE endpoint on whichever type had capacity, so they cannot
# produce a three-way hardware comparison.
INSTANCE_POOLS = None

# Engine configuration. These four decide whether the model fits; see stage 3.
TENSOR_PARALLEL_SIZE = 4
DATA_PARALLEL_SIZE = 2            # attention replicas. EP = TP x DP
ENABLE_EXPERT_PARALLEL = True     # with TP=4, DP=2 this gives EP=8
MAX_MODEL_LEN = 32768
MAX_NUM_SEQS = 128                # per DP replica, so 256 total here
GPU_MEMORY_UTILIZATION = 0.85
MAX_NUM_BATCHED_TOKENS = 8192

session = boto3.Session(profile_name=PROFILE, region_name=REGION)
sm = session.client("sagemaker")
smr = session.client("sagemaker-runtime")
cw = session.client("cloudwatch")

ACCOUNT = session.client("sts").get_caller_identity()["Account"]
BUCKET = f"sagemaker-{REGION}-{ACCOUNT}"          # contains 'sagemaker', as required
MODEL_S3_URI = f"s3://{BUCKET}/models/glm-5.3-flash/"
IMAGE_URI = f"{ACCOUNT}.dkr.ecr.{REGION}.amazonaws.com/{ECR_REPO}:{IMAGE_TAG}"
ROLE = (
    f"arn:aws:iam::{ACCOUNT}:role/service-role/"
    "AmazonSageMakerServiceCatalogProductsExecutionRole"
)

print(f"account   {ACCOUNT}")
print(f"region    {REGION}")
print(f"image     {IMAGE_URI}")
print(f"weights   {MODEL_S3_URI}")
print(f"role      {ROLE}")

---
## 1. The model, measured

Before choosing hardware, know the weight footprint. These are the actual `safetensors`
dtype counts from the Hugging Face API, not an estimate.

| dtype | params | bytes each | size |
| --- | --- | --- | --- |
| `F8_E4M3` | 314.40 B | 1 | 292.8 GiB |
| `BF16` | 6.93 B | 2 | 12.9 GiB |
| `F32` | 295,518 | 4 | negligible |
| **total** | **321.3 B** | | **~306 GiB** |

Confirmed by the staging job, which moved **305.8 GiB across 62 shards (72 files)**.

**306 GiB of weights is why this needs 8 GPUs, and it is the floor.** It has nothing to do
with how much throughput you want. There is no 4-GPU H200 instance on SageMaker or EC2, so
an 8-GPU shape is the minimum viable deployment even for modest traffic.

Architecture facts from `config.json` that drive the serving config:

- 45 layers: **34 KDA linear-attention** + **11 DeepSeek-sparse-attention (MLA)**, plus 1 MTP layer
- MoE: **288 routed experts**, top-8 + 1 shared, `moe_intermediate_size` 2048, 42 sparse layers
- MLA: `kv_lora_rank` 512, **`qk_rope_head_dim` 0 (NoPE)** <- this one bites later
- `max_position_embeddings` 1,048,576
- Natively multimodal (`Glm5NextForConditionalGeneration`), vision tower included

Run the next cell to verify the footprint yourself rather than trusting the table.

In [ ]:
import urllib.request

BYTES_PER_DTYPE = {"F8_E4M3": 1, "F8_E5M2": 1, "BF16": 2, "F16": 2, "F32": 4, "I64": 8}

req = urllib.request.Request(f"https://huggingface.co/api/models/{HF_MODEL_ID}")
with urllib.request.urlopen(req, timeout=30) as resp:
    info = json.load(resp)

params = info.get("safetensors", {}).get("parameters", {})
total_bytes = 0
print(f"{'dtype':<10}{'params':>18}{'GiB':>10}")
for dtype, count in sorted(params.items()):
    width = BYTES_PER_DTYPE.get(dtype, 2)
    size = count * width
    total_bytes += size
    print(f"{dtype:<10}{count:>18,}{size / 2**30:>10,.1f}")

total_params = sum(params.values())
print(f"\n{'total':<10}{total_params:>18,}{total_bytes / 2**30:>10,.1f}")
print(f"\nper GPU at 8-way sharding: {total_bytes / 2**30 / 8:,.1f} GiB")

---
## 2. Stage 1 and 2: container and weights

Both are one-time. Both run outside this notebook because they need Docker and a Processing
job respectively, and neither belongs in a notebook kernel.

### 2.1 The container

GLM-5.3-Flash is not in any tagged vLLM release, so the image is a thin layer over the
purpose-built dev image plus a `serve` shim:

```dockerfile
ARG BASE_IMAGE=vllm/vllm-openai:glm53-flash
FROM ${BASE_IMAGE}
COPY serve /usr/bin/serve
RUN chmod 777 /usr/bin/serve
ENV HF_HOME=/tmp/hf TOKENIZERS_PARALLELISM=false
ENTRYPOINT [ "/usr/bin/serve" ]
```

`serve` exists to satisfy the SageMaker hosting contract (listen on 8080, expose `/ping` and
`/invocations`) and to translate environment variables into engine flags. It reads every
`SM_VLLM_<FLAG>` variable and emits `--<flag>`, with `SM_VLLM_MODEL` becoming the positional
argument. **That means all engine tuning happens through the Model's `Environment` map with
no image rebuild.** vLLM's OpenAI server already provides `/ping` and `/invocations`.

Build it with CodeBuild if you have no local Docker:

```bash
# Section 2.4-2.5 below do this inline. This CLI equivalent is optional.
python prep_glm53_flash.py build-image --profile <your-profile> --region <your-region>
```

Produces roughly an 8.6 GB image. Measured build time was 618 s.

### 2.2 The weights, and why S3 staging is mandatory

**A bare Hugging Face repo id cannot work with this model.** vLLM's multimodal processor for
this architecture does a literal local open:

```python
# vllm/transformers_utils/processors/glm5next.py
with open(os.path.join(model_path, "processor_config.json")) as f:
```

There is no Hub resolution. Passing `zai-org/GLM-5.3-Flash` makes it look for a *relative
directory* of that name and die during multimodal profiling, long before any weights load:

```
FileNotFoundError: [Errno 2] No such file or directory:
  'zai-org/GLM-5.3-Flash/processor_config.json'
```

The file does exist in the repo (909 bytes). It just has to be on disk. So the weights must
be an S3 prefix mounted at `/opt/ml/model`, never a repo id. The container crash-loops on
this until the health check expires, so you pay the full startup timeout to learn it.

```bash
# Section 2.6 below does this inline. This CLI equivalent is optional.
python prep_glm53_flash.py stage-weights --profile <your-profile> --region <your-region> \
    --sagemaker-role-arn <role-with-AmazonSageMakerFullAccess> \
    --instance-type ml.m5.12xlarge --max-workers 16 --volume-size-gb 30
```

**Stream, do not download-then-upload.** `huggingface_hub` routes large files through Xet,
and a `token=` argument passed per call does not reliably reach that layer:

| approach | result |
| --- | --- |
| `hf_hub_download` (Xet), unauthenticated | 0 of 72 files in 7 min |
| `hf_hub_download` (Xet), token per call | 12 of 72, then stalled 15+ min |
| streamed `resolve/` + S3 multipart, token in env | **72 of 72, 305.8 GiB, 14.8 min, 353 MiB/s** |

Also set `AWS_DEFAULT_REGION` in the Processing job environment. Processing containers have
no implicit region and every boto3 client otherwise raises `NoRegionError`.

The next cell just verifies staging completed.

### 2.3 Preflight: what is already done

Both remaining stages are one-time and both cost money, so this checks state first and
each stage below is gated behind its own explicit flag.

Note on the ECR check: the CodeBuild service role is **not** granted `ecr:DescribeImages`,
so a post-push verification call can return `AccessDenied` even though the push succeeded.
Keep such checks non-fatal.


In [ ]:
# Both stages below are implemented inline in this notebook; nothing external is
# needed. This cell only decides which of them still has work to do.

ecr = session.client("ecr")


def image_exists():
    try:
        ecr.describe_images(
            repositoryName=ECR_REPO, imageIds=[{"imageTag": IMAGE_TAG}]
        )
        return True
    except (ecr.exceptions.RepositoryNotFoundException,
            ecr.exceptions.ImageNotFoundException):
        return False
    except botocore.exceptions.ClientError as exc:
        if exc.response["Error"]["Code"] in ("AccessDeniedException", "AccessDenied"):
            print("  ecr:DescribeImages denied, cannot verify; assuming present")
            return True
        raise


def weights_staged(expected=72):
    prefix = MODEL_S3_URI.replace(f"s3://{BUCKET}/", "")
    resp = session.client("s3").list_objects_v2(Bucket=BUCKET, Prefix=prefix)
    return resp.get("KeyCount", 0) >= expected


NEED_BUILD = not image_exists()
NEED_STAGE = not weights_staged()

print(f"container image   {'MISSING, build needed' if NEED_BUILD else 'present'}")
print(f"staged weights    {'MISSING, staging needed' if NEED_STAGE else 'present'}")
if NEED_BUILD or NEED_STAGE:
    print("\nSet RUN_BUILD / RUN_STAGE to True below to run the missing stage(s).")
else:
    print("\nBoth artifacts exist. Sections 2.4-2.6 will self-skip; go to stage 3.")

### 2.4 Write the container files

Three files define the image. Writing them from the notebook keeps it standalone; if you have
the repo, these are identical to `Dockerfile`, `serve`, and `prep/buildspec.yml`.

`serve` is the whole SageMaker integration. It turns every `SM_VLLM_<FLAG>` environment
variable into `--<flag>`, and `SM_VLLM_MODEL` into the positional argument, then execs
`vllm serve`. That is why every engine change in this notebook is an environment edit and
never an image rebuild.

In [ ]:
import pathlib

BUILD_DIR = pathlib.Path("build")
BUILD_DIR.mkdir(exist_ok=True)

DOCKERFILE = r"""
# GLM-5.3-Flash (Glm5NextForConditionalGeneration) is not in any tagged vLLM release,
# so the base is the purpose-built dev image.
ARG BASE_IMAGE=vllm/vllm-openai:glm53-flash
FROM ${BASE_IMAGE}

# SageMaker hosting contract: listen on 8080, expose /ping and /invocations.
# vLLM's OpenAI server provides both; serve only translates env vars to CLI flags.
COPY serve /usr/bin/serve
RUN chmod 777 /usr/bin/serve

# HF_HOME must be writable and off the small container layer.
ENV HF_HOME=/tmp/hf \
    TOKENIZERS_PARALLELISM=false

ENTRYPOINT [ "/usr/bin/serve" ]
"""

SERVE = r"""#!/bin/bash
PREFIX="SM_VLLM_"
ARG_PREFIX="--"

# The model is the first positional arg to `vllm serve`; skip it in the loop below.
MODEL_VAR="${PREFIX}MODEL"
MODEL="${!MODEL_VAR}"

# Port 8080 is required by SageMaker.
PORT=(--port 8080)
ARGS=("${PORT[@]}")

while IFS='=' read -r key value; do
    if [ "$key" = "${PREFIX}MODEL" ]; then
        continue
    fi
    # SM_VLLM_MAX_MODEL_LEN -> --max-model-len
    arg_name=$(echo "${key#"${PREFIX}"}" | tr '[:upper:]' '[:lower:]' | tr '_' '-')
    ARGS+=("${ARG_PREFIX}${arg_name}")
    # Bare flags carry no value; "true" means store_true.
    if [ -n "$value" ] && [ "$value" != "true" ] && [ "$value" != "True" ]; then
        ARGS+=("$value")
    fi
done < <(env | grep "^${PREFIX}")

echo "-------------------------------------------------------------------"
echo "vLLM model: [${MODEL}]"
echo "vLLM engine args: [${ARGS[@]}]"
echo "-------------------------------------------------------------------"

if [ -n "$MODEL" ]; then
    exec vllm serve "$MODEL" "${ARGS[@]}"
else
    exec vllm serve "${ARGS[@]}"
fi
"""

BUILDSPEC = r"""
version: 0.2

env:
  variables:
    DOCKER_BUILDKIT: "1"

phases:
  pre_build:
    commands:
      - IMAGE_URI="${AWS_ACCOUNT_ID}.dkr.ecr.${AWS_DEFAULT_REGION}.amazonaws.com/${REPOSITORY_NAME}:${IMAGE_TAG}"
      - echo "target=$IMAGE_URI base=$BASE_IMAGE"
      - df -h /
      - aws ecr describe-repositories --repository-names "$REPOSITORY_NAME" --region "$AWS_DEFAULT_REGION"
        || aws ecr create-repository --repository-name "$REPOSITORY_NAME" --region "$AWS_DEFAULT_REGION"
      - aws ecr get-login-password --region "$AWS_DEFAULT_REGION"
        | docker login --username AWS --password-stdin "${AWS_ACCOUNT_ID}.dkr.ecr.${AWS_DEFAULT_REGION}.amazonaws.com"
      - |
        if [ -n "$DOCKERHUB_USER" ] && [ -n "$DOCKERHUB_TOKEN" ]; then
          echo "$DOCKERHUB_TOKEN" | docker login --username "$DOCKERHUB_USER" --password-stdin
        else
          echo "no Docker Hub creds provided; pulling $BASE_IMAGE anonymously"
        fi

  build:
    commands:
      # --platform is explicit: the base tag is a multi-arch manifest and the target is x86.
      - docker build --platform linux/amd64 --build-arg "BASE_IMAGE=${BASE_IMAGE}" --file Dockerfile --tag "$IMAGE_URI" .
      # Fail here, loudly, rather than at endpoint creation if serve did not land.
      - docker run --rm --entrypoint /bin/sh "$IMAGE_URI" -c "test -x /usr/bin/serve && head -1 /usr/bin/serve"
      - docker run --rm --entrypoint /bin/sh "$IMAGE_URI" -c "python3 -c 'import vllm; print(\"vllm\", vllm.__version__)'"

  post_build:
    commands:
      - docker push "$IMAGE_URI"
      # Deliberately non-fatal: the SageMaker CodeBuild service role is not granted
      # ecr:DescribeImages, so this returns 254 even on a fully successful push.
      - aws ecr describe-images --repository-name "$REPOSITORY_NAME" --image-ids "imageTag=$IMAGE_TAG" --region "$AWS_DEFAULT_REGION" || echo "(describe-images not permitted; the push itself succeeded)"
      - echo "pushed $IMAGE_URI"
"""

(BUILD_DIR / "Dockerfile").write_text(DOCKERFILE.lstrip(), encoding="utf-8")
(BUILD_DIR / "serve").write_text(SERVE, encoding="utf-8", newline="\n")
(BUILD_DIR / "buildspec.yml").write_text(BUILDSPEC.lstrip(), encoding="utf-8")

for path in sorted(BUILD_DIR.iterdir()):
    print(f"  {path}  {path.stat().st_size:>6,} bytes")
print("\nNote: serve is written with LF endings on purpose. CRLF in a shell script")
print("gives '/usr/bin/serve: cannot execute: required file not found' in the container.")

### 2.5 Build and push the image with CodeBuild

No local Docker needed. The notebook zips the three files to S3, creates a CodeBuild project
with `privilegedMode` (required to run a Docker daemon), and starts a build.

`BUILD_GENERAL1_LARGE` is not optional: the compressed base image is about 8.6 GB and smaller
compute types run out of disk. Measured build time was 618 s.

Remember the ECR repository must be named `sagemaker-*`, or the CodeBuild service role's
policy will refuse `ecr:InitiateLayerUpload`.

In [ ]:
import io
import zipfile

RUN_BUILD = False    # flip to True to actually build; a build bills CodeBuild minutes

CODEBUILD_PROJECT = "glm53-flash-image"
BASE_IMAGE = "vllm/vllm-openai:glm53-flash"
CODEBUILD_ROLE = (
    f"arn:aws:iam::{ACCOUNT}:role/service-role/"
    "AmazonSageMakerServiceCatalogProductsCodeBuildRole"
)

cb = session.client("codebuild")


def upload_source_bundle():
    """Zip Dockerfile + serve + buildspec.yml and put it in S3 as the build source."""
    buffer = io.BytesIO()
    with zipfile.ZipFile(buffer, "w", zipfile.ZIP_DEFLATED) as archive:
        for name in ("Dockerfile", "serve", "buildspec.yml"):
            archive.write(BUILD_DIR / name, arcname=name)
    key = f"codebuild/glm53-flash/{int(time.time())}/source.zip"
    session.client("s3").put_object(Bucket=BUCKET, Key=key, Body=buffer.getvalue())
    print(f"source bundle  s3://{BUCKET}/{key}  ({buffer.tell():,} bytes)")
    return key


def ensure_project(source_key):
    spec = {
        "name": CODEBUILD_PROJECT,
        "source": {
            "type": "S3",
            "location": f"{BUCKET}/{source_key}",
            "buildspec": "buildspec.yml",
        },
        "artifacts": {"type": "NO_ARTIFACTS"},
        "environment": {
            "type": "LINUX_CONTAINER",
            "image": "aws/codebuild/standard:7.0",
            "computeType": "BUILD_GENERAL1_LARGE",   # disk headroom for the 8.6 GB base
            "privilegedMode": True,                  # required: we run a Docker daemon
            "environmentVariables": [
                {"name": "AWS_ACCOUNT_ID", "value": ACCOUNT},
                {"name": "REPOSITORY_NAME", "value": ECR_REPO},
                {"name": "IMAGE_TAG", "value": IMAGE_TAG},
                {"name": "BASE_IMAGE", "value": BASE_IMAGE},
            ],
        },
        "serviceRole": CODEBUILD_ROLE,
        "timeoutInMinutes": 60,
    }
    try:
        cb.create_project(**spec)
        print(f"created CodeBuild project {CODEBUILD_PROJECT}")
    except cb.exceptions.ResourceAlreadyExistsException:
        cb.update_project(**spec)
        print(f"updated CodeBuild project {CODEBUILD_PROJECT}")


def run_build():
    build_id = cb.start_build(projectName=CODEBUILD_PROJECT)["build"]["id"]
    print(f"build started   {build_id}")
    started, last = time.time(), None
    while True:
        build = cb.batch_get_builds(ids=[build_id])["builds"][0]
        status, phase = build["buildStatus"], build.get("currentPhase")
        if (status, phase) != last:
            print(f"  [{time.time() - started:>5.0f}s] {status} {phase}")
            last = (status, phase)
        if status != "IN_PROGRESS":
            return status
        time.sleep(20)


if not RUN_BUILD:
    print(f"RUN_BUILD is False. Image needed: {NEED_BUILD}")
elif not NEED_BUILD:
    print("image already present, skipping build")
else:
    ensure_project(upload_source_bundle())
    print(f"\nfinal status: {run_build()}")
    print(f"image: {IMAGE_URI}")

### 2.6 Stage the weights into S3 with a Processing job

The worker streams each file from Hugging Face straight into S3 multipart. It never lands a
file on disk, which is why a 30 GB volume can move 306 GiB.

Four details in the worker below are load-bearing, and each one cost real debugging time:

- **Bypass Xet.** `huggingface_hub` routes large files through Xet, a `token=` argument passed
  per call does not reliably reach that layer, and unauthenticated Xet throttles to a
  standstill. Fetching `resolve/<rev>/<file>` with an `Authorization` header sidesteps it.
- **`AWS_DEFAULT_REGION` in the job environment.** Processing containers have no implicit
  region and every boto3 client otherwise raises `NoRegionError`.
- **`RLock`, not `Lock`.** A non-reentrant lock acquired twice on one thread deadlocks that
  thread while it holds the lock, wedging every other worker. The symptom was a job sitting in
  `InProgress` with zero uploads, zero completions and zero errors.
- **Only the secret *id* travels in the job definition.** The worker resolves the token at
  runtime, so `DescribeProcessingJob` never exposes it.

This is a compact equivalent of `prep/stage_weights.py`. The repo version has more retry and
resume logic; this one is complete enough to do the job.

In [ ]:
STAGER = r"""
import json
import os
import threading
import urllib.request
from concurrent.futures import ThreadPoolExecutor

import boto3

REPO = os.environ["HF_REPO"]
DEST = os.environ["S3_DEST"]
WORKERS = int(os.environ.get("MAX_WORKERS", "8"))
SECRET_ID = os.environ.get("HF_TOKEN_SECRET_ID") or ""

session = boto3.Session()
s3 = session.client("s3")

token = ""
if SECRET_ID:
    raw = session.client("secretsmanager").get_secret_value(
        SecretId=SECRET_ID
    )["SecretString"]
    try:
        token = json.loads(raw).get("token", raw)
    except json.JSONDecodeError:
        token = raw
    os.environ["HF_TOKEN"] = token

bucket, prefix = DEST[len("s3://"):].split("/", 1)

request = urllib.request.Request(f"https://huggingface.co/api/models/{REPO}")
if token:
    request.add_header("Authorization", "Bearer " + token)
with urllib.request.urlopen(request, timeout=60) as response:
    files = [s["rfilename"] for s in json.load(response)["siblings"]]
print("%d files to move" % len(files), flush=True)

# RLock, not Lock. See the notebook note above.
counter_lock = threading.RLock()
state = {"done": 0, "bytes": 0}


def move(name):
    url = "https://huggingface.co/%s/resolve/main/%s" % (REPO, name)
    req = urllib.request.Request(url)
    if token:
        req.add_header("Authorization", "Bearer " + token)
    # Stream into S3 multipart. boto3 handles the part splitting for a
    # non-seekable stream, so nothing is written to the local volume.
    with urllib.request.urlopen(req, timeout=1800) as body:
        size = int(body.headers.get("Content-Length") or 0)
        s3.upload_fileobj(body, bucket, prefix + name)
    with counter_lock:
        state["done"] += 1
        state["bytes"] += size
        print("  [%d/%d] %s (%.1f GiB total)"
              % (state["done"], len(files), name, state["bytes"] / 2**30),
              flush=True)


errors = []
with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    for name, result in zip(files, pool.map(move, files, timeout=None)):
        pass

print("staged %d/%d files, %.1f GiB, to %s"
      % (state["done"], len(files), state["bytes"] / 2**30, DEST), flush=True)
if state["done"] != len(files):
    raise SystemExit("incomplete: %d of %d" % (state["done"], len(files)))
"""

(BUILD_DIR / "stage_weights.py").write_text(STAGER.lstrip(), encoding="utf-8")
print(f"wrote {BUILD_DIR / 'stage_weights.py'}  "
      f"{(BUILD_DIR / 'stage_weights.py').stat().st_size:,} bytes")

import ast
ast.parse((BUILD_DIR / "stage_weights.py").read_text(encoding="utf-8"))
print("worker script parses")

In [ ]:
RUN_STAGE = False   # flip to True to launch; this bills an ml.m5.12xlarge Processing job

# Secrets Manager id holding {"token": "hf_..."}. Only the id travels in the job
# definition; the worker resolves the token at runtime. Must be named
# AmazonSageMaker-* or AmazonSageMakerFullAccess will not permit GetSecretValue.
HF_TOKEN_SECRET = "AmazonSageMaker-hf-token"
STAGE_INSTANCE = "ml.m5.12xlarge"
STAGE_WORKERS = 16
STAGE_VOLUME_GB = 30       # the job streams; it never lands the files


def launch_staging():
    job_name = f"glm53-flash-stage-{int(time.time())}"
    code_key = f"processing/glm53-flash/{job_name}/stage_weights.py"

    session.client("s3").put_object(
        Bucket=BUCKET,
        Key=code_key,
        Body=(BUILD_DIR / "stage_weights.py").read_bytes(),
    )

    sm.create_processing_job(
        ProcessingJobName=job_name,
        RoleArn=ROLE,
        AppSpecification={
            "ImageUri": IMAGE_URI,
            # Overrides the image's /usr/bin/serve entrypoint.
            "ContainerEntrypoint": [
                "python3", "/opt/ml/processing/input/code/stage_weights.py"
            ],
        },
        ProcessingInputs=[{
            "InputName": "code",
            "S3Input": {
                "S3Uri": f"s3://{BUCKET}/{code_key}",
                "LocalPath": "/opt/ml/processing/input/code",
                "S3DataType": "S3Prefix",
                "S3InputMode": "File",
                "S3DataDistributionType": "FullyReplicated",
            },
        }],
        ProcessingResources={"ClusterConfig": {
            "InstanceCount": 1,
            "InstanceType": STAGE_INSTANCE,
            "VolumeSizeInGB": STAGE_VOLUME_GB,
        }},
        StoppingCondition={"MaxRuntimeInSeconds": 6 * 3600},
        Environment={
            "HF_REPO": HF_MODEL_ID,
            "S3_DEST": MODEL_S3_URI,
            "MAX_WORKERS": str(STAGE_WORKERS),
            # Processing containers have no implicit region.
            "AWS_DEFAULT_REGION": REGION,
            # Only the secret id travels in the job definition, never the token.
            "HF_TOKEN_SECRET_ID": HF_TOKEN_SECRET,
        },
    )
    print(f"processing job started  {job_name}")
    print(f"  logs: /aws/sagemaker/ProcessingJobs -> {job_name}")

    started, last = time.time(), None
    while True:
        desc = sm.describe_processing_job(ProcessingJobName=job_name)
        status = desc["ProcessingJobStatus"]
        if status != last:
            print(f"  [{time.time() - started:>5.0f}s] {status}")
            last = status
        if status in ("Completed", "Failed", "Stopped"):
            if status != "Completed":
                print(f"  reason: {desc.get('FailureReason')}")
            return status
        time.sleep(30)


if not RUN_STAGE:
    print(f"RUN_STAGE is False. Weights needed: {NEED_STAGE}")
elif not NEED_STAGE:
    print("weights already staged, skipping")
else:
    print(f"final status: {launch_staging()}")
    print("measured on the real run: 305.8 GiB in 14.8 min at 353 MiB/s")

In [ ]:
s3 = session.client("s3")
prefix = MODEL_S3_URI.replace(f"s3://{BUCKET}/", "")

objects, token, total = [], None, 0
while True:
    kwargs = {"Bucket": BUCKET, "Prefix": prefix}
    if token:
        kwargs["ContinuationToken"] = token
    page = s3.list_objects_v2(**kwargs)
    for obj in page.get("Contents", []):
        objects.append(obj["Key"].rsplit("/", 1)[-1])
        total += obj["Size"]
    token = page.get("NextContinuationToken")
    if not token:
        break

print(f"objects        {len(objects)}")
print(f"total size     {total / 2**30:,.1f} GiB")

# The file whose absence costs you a full startup timeout to diagnose.
required = ["processor_config.json", "config.json", "tokenizer_config.json"]
for name in required:
    mark = "present" if name in objects else "MISSING"
    print(f"{name:<28}{mark}")

shards = [o for o in objects if o.endswith(".safetensors")]
print(f"safetensors shards           {len(shards)}")

---
## 3. Stage 3: deploy

### 3.1 Choosing the topology

**vLLM has no `--expert-parallel-size`.** EP width is derived: **EP = TP x DP**. So "TP=4,
EP=2" is not expressible. On 8 GPUs the EP config is TP=4, DP=2, `--enable-expert-parallel`,
which gives EP=8. Note `--max-num-seqs` is **per DP replica**, so DP=2 with 128 gives 256 total.

Per-GPU weight cost barely moves between topologies, so choose on throughput, not memory:

| topology | EP | experts/GPU | other/GPU | total/GPU | attention replicas |
| --- | --- | --- | --- | --- | --- |
| TP=8, DP=1 | off | 36.6 GiB | 1.7 GiB | 38.3 GiB | 1 |
| TP=4, DP=2, EP | 8 | 36.6 GiB | 3.3 GiB | 39.9 GiB | 2 |
| TP=2, DP=4, EP | 8 | 36.6 GiB | 6.6 GiB | 43.2 GiB | 4 |

**DP=2 is not two replicas.** The 288 routed experts shard exactly once across all 8 GPUs;
only attention and dense weights duplicate per DP group. It is one endpoint, one instance,
one copy of the expert weights. Setting DP=1 does not reduce the bill by a cent.

### 3.2 The most important sizing finding: MLA KV is replicated across TP ranks

Measured at TP=8: `82.07 GiB / 7,253,058 tokens = 11.86 KiB per token`, matching the estimate
derived from `config.json` to within 4%. Critically, the pool equals **one GPU's** KV memory,
not eight, because MLA caches a compressed latent that is not head-partitioned. Therefore:

- Raising **TP does not raise KV capacity**, it duplicates the same cache more times.
- Raising **DP does**, because each attention replica holds a distinct cache.

Confirmed a second way: DP=2 raised total usable KV from 7,253,058 to **9,857,994 tokens**
despite utilisation dropping from 0.90 to 0.85. Verify on any new config by dividing
`Available KV cache memory` by `GPU KV cache size` and checking whether the pool tracks one
GPU or all of them.

### 3.3 Two timeouts, both capped at 3600 s

- `ModelDataDownloadTimeoutInSeconds`, the S3 to instance copy
- `ContainerStartupHealthCheckTimeoutInSeconds`, engine init until `/ping` answers

Staging to S3 splits the work across two separate budgets instead of racing 306 GiB of
download *and* engine init inside one. Measured engine init was 404 to 514 s, mostly CUDA
graph capture, comfortably inside the second budget.

**Do not set `VolumeSizeInGB`.** The 8-GPU types all ship local NVMe and SageMaker rejects the
parameter for instance types that provide instance storage.

In [ ]:
CONTAINER_MODEL_DIR = "/opt/ml/model"
MAX_DOWNLOAD_TIMEOUT = 3600      # service cap
MAX_STARTUP_TIMEOUT = 3600       # service cap


def build_environment():
    """SM_VLLM_* becomes `vllm serve --<flag>`; everything else is plain container env."""
    env = {
        # positional model argument. A local directory, never a repo id (see 2.2)
        "SM_VLLM_MODEL": CONTAINER_MODEL_DIR,
        "SM_VLLM_SERVED_MODEL_NAME": SERVED_MODEL_NAME,
        # sharding
        "SM_VLLM_TENSOR_PARALLEL_SIZE": str(TENSOR_PARALLEL_SIZE),
        # the two knobs that decide whether this fits
        "SM_VLLM_MAX_MODEL_LEN": str(MAX_MODEL_LEN),
        "SM_VLLM_MAX_NUM_SEQS": str(MAX_NUM_SEQS),
        # memory and scheduling
        "SM_VLLM_GPU_MEMORY_UTILIZATION": str(GPU_MEMORY_UTILIZATION),
        "SM_VLLM_MAX_NUM_BATCHED_TOKENS": str(MAX_NUM_BATCHED_TOKENS),
        # bound the vision path so profiling does not reserve for the worst case
        "SM_VLLM_LIMIT_MM_PER_PROMPT": json.dumps({"image": 4, "video": 0}),
        # Keep off the fp8_ds_mla kernel. On 96 GiB cards the memory profiler can
        # exhaust its budget, fall back to an FP8 KV cache, and assert
        # 'pe_dim must be 64 for fp8_ds_mla'. This model is NoPE MLA
        # (qk_rope_head_dim=0) so pe_dim is 0 and that kernel cannot serve it.
        "SM_VLLM_KV_CACHE_DTYPE": "auto",
        # plain container env, no SM_VLLM_ prefix so not passed to vllm
        "HF_HOME": "/tmp/hf",
        "TOKENIZERS_PARALLELISM": "false",
    }
    if DATA_PARALLEL_SIZE > 1:
        env["SM_VLLM_DATA_PARALLEL_SIZE"] = str(DATA_PARALLEL_SIZE)
    if ENABLE_EXPERT_PARALLEL:
        env["SM_VLLM_ENABLE_EXPERT_PARALLEL"] = "true"
    return env


def render_vllm_command(env):
    """Reproduce what `serve` will exec. Review this before spending an hour of GPU time."""
    parts = ["vllm serve", env["SM_VLLM_MODEL"], "--port 8080"]
    for key, value in env.items():
        if not key.startswith("SM_VLLM_") or key == "SM_VLLM_MODEL":
            continue
        flag = "--" + key[len("SM_VLLM_"):].lower().replace("_", "-")
        if value in ("true", "True"):
            parts.append(flag)                    # store_true flags take no value
        elif any(ch in value for ch in ' {}"'):
            parts.append(f"{flag} '{value}'")     # e.g. --limit-mm-per-prompt JSON
        else:
            parts.append(f"{flag} {value}")
    return " \\\n    ".join(parts)


ENV = build_environment()
print(render_vllm_command(ENV))

gib_per_gpu = 306 / 8
print(f"\nweights per GPU     {gib_per_gpu:,.1f} GiB")
print(f"EP width            {TENSOR_PARALLEL_SIZE * DATA_PARALLEL_SIZE if ENABLE_EXPERT_PARALLEL else 'off'}")
print(f"total max_num_seqs  {MAX_NUM_SEQS * DATA_PARALLEL_SIZE}")

### 3.4 Create the model, endpoint config, and endpoint

`CompressionType: "None"` matters. The weights are already uncompressed shards; SageMaker
mounts the prefix at `/opt/ml/model` rather than trying to expand an archive.

`RoutingConfig` is set to `LEAST_OUTSTANDING_REQUESTS` because LLM requests have wildly
uneven service times and random routing sends work to already-busy replicas.

In [ ]:
suffix = f"{int(time.time())}"
label = INSTANCE_TYPE.replace("ml.", "").replace(".", "-")
MODEL_NAME = f"glm53flash-{label}-{suffix}"
CONFIG_NAME = f"glm53flash-{label}-epc-{suffix}"
ENDPOINT_NAME = f"glm53flash-{label}-ep-{suffix}"

sm.create_model(
    ModelName=MODEL_NAME,
    ExecutionRoleArn=ROLE,
    PrimaryContainer={
        "Image": IMAGE_URI,
        "Environment": ENV,
        "ModelDataSource": {
            "S3DataSource": {
                "S3Uri": MODEL_S3_URI,
                "S3DataType": "S3Prefix",
                "CompressionType": "None",
            }
        },
    },
)
print(f"model created            {MODEL_NAME}")

variant = {
    "VariantName": "AllTraffic",
    "ModelName": MODEL_NAME,
    "InitialInstanceCount": 1,
    "InitialVariantWeight": 1.0,
    "ModelDataDownloadTimeoutInSeconds": MAX_DOWNLOAD_TIMEOUT,
    "ContainerStartupHealthCheckTimeoutInSeconds": MAX_STARTUP_TIMEOUT,
    "RoutingConfig": {"RoutingStrategy": "LEAST_OUTSTANDING_REQUESTS"},
    # Deliberately no VolumeSizeInGB: these types ship local NVMe and SageMaker
    # rejects the parameter for instance types that provide instance storage.
}

if INSTANCE_POOLS:
    # Ordered fallback, up to 5 types. Replaces InstanceType; setting both is invalid.
    variant["InstancePools"] = [
        {"InstanceType": t, "Priority": i} for i, t in enumerate(INSTANCE_POOLS)
    ]
    variant["VariantInstanceProvisionTimeoutInSeconds"] = 3600   # 300-3600
else:
    variant["InstanceType"] = INSTANCE_TYPE

sm.create_endpoint_config(EndpointConfigName=CONFIG_NAME, ProductionVariants=[variant])
print(f"endpoint config created  {CONFIG_NAME}")

sm.create_endpoint(EndpointName=ENDPOINT_NAME, EndpointConfigName=CONFIG_NAME)
print(f"endpoint creating        {ENDPOINT_NAME}")
print(f"logs                     /aws/sagemaker/Endpoints/{ENDPOINT_NAME}")

### 3.5 Wait, without destroying the evidence

Two hard-won rules in this poll loop:

**Never let a transient error trigger a teardown.** In this project a broad `except Exception`
once caught a `UnicodeEncodeError` from a check-mark emoji on a cp1252 console, concluded the
deploy had failed, and **deleted a healthy, fully warmed endpoint that had already passed its
smoke test**. A later run lost an attempt to a transient `EndpointConnectionError` mid-poll
the same way. So this loop retries connection errors and never deletes anything.

**Never delete a still-`Creating` endpoint.** Doing so destroys its `FailureReason`, which is
the only thing that tells you whether you hit capacity or a boot failure. Leave it and read it.

Expect roughly 15 minutes on a good run. `InsufficientInstanceCapacity` takes about 31 minutes
to surface. Note that `VariantInstanceProvisionTimeoutInSeconds` does **not** fast-fail a
pinned single-instance-type variant; it appears to apply to instance pools only.

In [ ]:
def wait_for_endpoint(name, timeout=3900, poll=30):
    """Poll to a terminal state. Returns the final description. Deletes nothing, ever."""
    started = time.time()
    transient = 0
    while time.time() - started < timeout:
        try:
            desc = sm.describe_endpoint(EndpointName=name)
            transient = 0
        except (botocore.exceptions.EndpointConnectionError,
                botocore.exceptions.ConnectionClosedError,
                botocore.exceptions.ReadTimeoutError) as exc:
            # A local network blip is not a deploy failure. Keep polling.
            transient += 1
            print(f"  [{time.time() - started:>5.0f}s] transient {type(exc).__name__} "
                  f"(#{transient}), retrying")
            time.sleep(poll)
            continue

        status = desc["EndpointStatus"]
        print(f"  [{time.time() - started:>5.0f}s] {status}")
        if status in ("InService", "Failed", "OutOfService"):
            if status != "InService":
                print(f"\nFailureReason: {desc.get('FailureReason', '(none reported)')}")
            return desc
        time.sleep(poll)

    print("\npoll timeout. The endpoint is left in place deliberately, so its")
    print("FailureReason survives. Check DescribeEndpoint and CloudWatch.")
    return sm.describe_endpoint(EndpointName=name)


desc = wait_for_endpoint(ENDPOINT_NAME)

if desc["EndpointStatus"] == "InService":
    variants = desc["ProductionVariants"][0]
    # With InstancePools, this is how you learn which type actually got capacity.
    print(f"\nserving on {variants.get('InstancePools') or INSTANCE_TYPE}")

### 3.6 Confirm the topology and KV pool from the engine log

Do not trust the config, read what the engine actually did. vLLM's worker names confirm the
topology: `Worker_DP0_TP0_EP0` through `Worker_DP1_TP3_EP7` for TP=4/DP=2/EP=8.

Measured on that config:

```
Available KV cache memory: 71.31 GiB            (per GPU)
GPU KV cache size: 4,928,997 tokens             (per DP replica; x2 = 9,857,994)
Maximum concurrency for 32,768 tokens per request: 150.42x
init engine (profile, create kv cache, warmup model) took 404.4 s
```

In [ ]:
logs = session.client("logs")
group = f"/aws/sagemaker/Endpoints/{ENDPOINT_NAME}"

PATTERNS = [
    "Loading weights took",
    "Available KV cache memory",
    "GPU KV cache size",
    "Maximum concurrency",
    "init engine",
    "Application startup complete",
]

for pattern in PATTERNS:
    try:
        events = logs.filter_log_events(
            logGroupName=group, filterPattern=f'"{pattern}"', limit=3
        )["events"]
    except logs.exceptions.ResourceNotFoundException:
        print(f"log group not found yet: {group}")
        break
    for event in events:
        print(event["message"].strip())

# The check that generalises: does the KV pool track one GPU or all of them?
print("\nDivide 'Available KV cache memory' by 'GPU KV cache size' for KiB/token.")
print("Measured 11.86 KiB/token at TP=8, matching the config.json estimate to 4%.")

---
## 4. Stage 4: smoke test

The container speaks the OpenAI chat completions API, so the `/invocations` payload is an
OpenAI request body.

**GLM-5.3-Flash defaults to `reasoning_effort=max`,** so responses open with reasoning tokens.
Any throughput number you measure counts thinking tokens, and TTFT is time to the first
*reasoning* token. Keep that in mind for every figure below.

In [ ]:
payload = {
    "model": SERVED_MODEL_NAME,
    "messages": [{"role": "user", "content": "In one sentence, what is MLA attention?"}],
    "max_tokens": 128,
    "temperature": 0.0,
}

started = time.perf_counter()
resp = smr.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType="application/json",
    Body=json.dumps(payload),
)
elapsed = time.perf_counter() - started
body = json.loads(resp["Body"].read())

print(f"round trip     {elapsed * 1000:,.0f} ms")
print(f"usage          {body.get('usage')}")
print(f"\n{body['choices'][0]['message']['content']}")

In [ ]:
# Streaming, which is how you get a real TTFT. Measured 232 ms on the p5en bring-up.
payload_stream = dict(payload, stream=True)

started = time.perf_counter()
ttft = None
chunks = 0
text = []

resp = smr.invoke_endpoint_with_response_stream(
    EndpointName=ENDPOINT_NAME,
    ContentType="application/json",
    Body=json.dumps(payload_stream),
)

for event in resp["Body"]:
    if "PayloadPart" not in event:
        continue
    for line in event["PayloadPart"]["Bytes"].decode("utf-8").splitlines():
        if not line.startswith("data: "):
            continue
        data = line[len("data: "):].strip()
        if data == "[DONE]":
            continue
        delta = json.loads(data)["choices"][0].get("delta", {})
        piece = delta.get("content") or delta.get("reasoning_content") or ""
        if piece:
            if ttft is None:
                ttft = time.perf_counter() - started
            chunks += 1
            text.append(piece)

total = time.perf_counter() - started
print(f"TTFT           {ttft * 1000:,.0f} ms" if ttft else "no tokens received")
print(f"chunks         {chunks}")
print(f"total          {total:,.1f} s")
print(f"\n{''.join(text)[:600]}")

---
## 5. Benchmark a concurrency sweep

Both paths below use **Amazon SageMaker AI optimized generative AI benchmarking**. The
difference is how you drive it: the Python SDK convenience wrapper, or the underlying
`CreateAIBenchmarkJob` API directly. The raw API is the better one and it is not obvious from
the docs why:

| | `sagemaker.serve.start_benchmark` wrapper | raw `CreateAIBenchmarkJob` |
| --- | --- | --- |
| concurrency argument | one value per job | **a list, one job sweeps everything** |
| wall clock for 5 points | ~17 min *per point* | **21 min total** |
| results | returned by the SDK, lost if the job is deleted | **persist in S3 independently** |
| percentiles | p50 and avg | **plus e2e latency percentiles** |

Four bugs cost real money here before they were fixed, and all four are worth avoiding:

1. **Write each point as it completes.** An incomplete sweep once persisted nothing, so 16 h of
   endpoint time (roughly $1,070 to $1,360) produced two numbers that existed only in a log.
2. **Bound the teardown.** `job.delete()` on a still-running job blocked for about 11.3 h.
3. **Send warm-up requests.** Without them the first point absorbs cold start and reads *worse*
   than the second, which looks like a hardware anomaly and is not.
4. **Keep the jobs.** Deleting them discards p90/p99, leaving only p50 and avg.

The repo wraps all of this:

```bash
python sweep_glm53_flash.py sweep --profile <your-profile> --region <your-region> \
    --endpoint-name <ep> --instance-label p5en-ep8 \
    --concurrency 16,32,64,128,256 --input-tokens 8192 --output-tokens 256
```

### Measured results, TP=4 / DP=2 / EP=8 on `ml.p5en.48xlarge`

8192 in / 256 out, streaming, `max_model_len` 32768, util 0.85:

| conc | TTFT p50 | TTFT p90 | TTFT p99 | ITL p50 | out tok/s | req/s |
| --- | --- | --- | --- | --- | --- | --- |
| 16 | 1428 | 12316 | 16940 | 20.1 | 311 | 1.2 |
| 32 | 979 | 4672 | 12443 | 25.0 | 842 | 3.3 |
| 64 | 634 | 2724 | 11327 | 32.3 | 1458 | 5.7 |
| 128 | **415** | 1561 | 17726 | 43.4 | **2359** | 9.3 |

Times in ms. TTFT p50 falls monotonically as concurrency rises, ITL climbs as decode contends,
throughput scales 311 to 2359 tok/s.

**The tail is the interesting part.** At concurrency 16, TTFT p50 1428 ms against p90 12,316 ms
is a 9x spread, and it is not cold start because warm-up ran. It is head-of-line blocking: with
8192-token inputs and `--max-num-batched-tokens 8192`, only one prefill fits per scheduler step,
so a queued request waits whole prefill cycles. If your traffic sits at low concurrency, that
tail is your user experience, and raising `--max-num-batched-tokens` is the lever to test.

---
## 6. GPU utilisation, and the mistake almost everyone makes

You do not need a special harness for this. SageMaker publishes GPU telemetry to CloudWatch
for **any** endpoint, so it can be recovered for any window, including a benchmark that has
already finished, with no re-run.

**`GPUUtilization` is summed across GPUs.** The ceiling on an 8-GPU instance is 800%, not 100%.
`CPUUtilization` is likewise summed across vCPUs. Divide, or the numbers look impossible.

In [ ]:
from datetime import datetime, timedelta, timezone

GPUS = 8
end = datetime.now(timezone.utc)
start = end - timedelta(minutes=30)

for metric in ("GPUUtilization", "GPUMemoryUtilization"):
    points = cw.get_metric_statistics(
        Namespace="/aws/sagemaker/Endpoints",
        MetricName=metric,
        Dimensions=[
            {"Name": "EndpointName", "Value": ENDPOINT_NAME},
            {"Name": "VariantName", "Value": "AllTraffic"},
        ],
        StartTime=start,
        EndTime=end,
        Period=60,
        Statistics=["Average", "Maximum"],
    )["Datapoints"]

    if not points:
        print(f"{metric}: no datapoints yet")
        continue

    avg = sum(p["Average"] for p in points) / len(points)
    peak = max(p["Maximum"] for p in points)
    print(f"{metric:<24} raw avg {avg:>7.1f}%   per-GPU avg {avg / GPUS:>5.1f}%"
          f"   per-GPU max {peak / GPUS:>5.1f}%")

### What this measurement corrected

The sweep results in section 5 show throughput still climbing at concurrency 128 with no knee,
which reads like spare capacity. It is not. Over a 20.2 min window covering all five levels:

| metric | raw avg | raw max | per-GPU avg | per-GPU max |
| --- | --- | --- | --- | --- |
| GPUUtilization | 572.6% | 797.0% | **71.6%** | **99.6%** |
| GPUMemoryUtilization | 774.3% | 774.3% | **96.8%** | 96.8% |

Per minute, GPUs were at **94 to 99% from the first minute of load, at every concurrency level
including 16**. There was never compute headroom. The 71.6% average is only the idle startup
minutes pulling it down. The gain from 311 to 2359 tok/s came from better batching efficiency
on already-busy GPUs, not from filling idle silicon.

This matters for cost conversations. "We only use 12% of peak throughput" describes batch
occupancy, not idle hardware, so there is no 88% to reclaim by shrinking the deployment.

Two caveats. A 60 s CloudWatch period against roughly 4 min levels blurs boundaries, so
individual minutes cannot be cleanly attributed to a concurrency level. And `GPUUtilization`
measures whether kernels are resident, not whether they do useful work; 98% is consistent with
busy-but-inefficient narrow GEMMs. `GPUMemoryUtilization` is flat because vLLM pre-allocates
the KV pool at startup, so it reflects `--gpu-memory-utilization`, not load.

---
## 7. Cost per token

One Pricing API trap: **SageMaker uses the `instanceName` attribute, not `instanceType`.**
Filtering on `instanceType` returns zero results **with no error**, which looks exactly like
"no price exists for this instance".

Also note the Pricing API lives in `us-east-1` regardless of which region you are pricing.

In [ ]:
pricing = session.client("pricing", region_name="us-east-1")


def hourly_rate(instance_type, location="US East (Ohio)"):
    """Real-time endpoint hosting rate. Note: instanceName, NOT instanceType."""
    pages = pricing.get_paginator("get_products").paginate(
        ServiceCode="AmazonSageMaker",
        Filters=[
            {"Type": "TERM_MATCH", "Field": "instanceName", "Value": instance_type},
            {"Type": "TERM_MATCH", "Field": "location", "Value": location},
        ],
    )
    for page in pages:
        for raw in page["PriceList"]:
            item = json.loads(raw)
            usage = item["product"]["attributes"].get("usagetype", "")
            if "Host:" not in usage:            # Host = real-time endpoint
                continue
            for term in item["terms"]["OnDemand"].values():
                for dim in term["priceDimensions"].values():
                    return float(dim["pricePerUnit"]["USD"]), usage
    return None, None


rate, usage = hourly_rate(INSTANCE_TYPE)
print(f"{INSTANCE_TYPE}  ${rate:,.4f}/hr  ({usage})\n")

# Cost at each measured concurrency level. Output-only is the number a customer feels;
# total includes prompt tokens, which are cheap to process but real.
MEASURED = {16: 311, 32: 842, 64: 1458, 128: 2359}   # concurrency -> output tok/s
INPUT_TOKENS, OUTPUT_TOKENS = 8192, 256

print(f"{'conc':>6}{'out tok/s':>12}{'$/1M out':>12}{'$/1M total':>13}")
for conc, tps in MEASURED.items():
    per_hour = tps * 3600
    dollars_out = rate / (per_hour / 1e6)
    total_tps = tps * (INPUT_TOKENS + OUTPUT_TOKENS) / OUTPUT_TOKENS
    dollars_total = rate / (total_tps * 3600 / 1e6)
    print(f"{conc:>6}{tps:>12,}{dollars_out:>12,.2f}{dollars_total:>13,.3f}")

print("\nThese assume a continuously loaded endpoint, which the section 6 telemetry")
print("supports while under load. Over an endpoint's whole life, duty cycle governs")
print("real cost per token far more than any engine setting does.")

---
## 8. Clean up

An idle endpoint bills at the full hourly rate. Delete in order: endpoint, then config, then
model.

In [ ]:
for delete, name, kwarg in (
    (sm.delete_endpoint, ENDPOINT_NAME, "EndpointName"),
    (sm.delete_endpoint_config, CONFIG_NAME, "EndpointConfigName"),
    (sm.delete_model, MODEL_NAME, "ModelName"),
):
    try:
        delete(**{kwarg: name})
        print(f"deleted {name}")
    except botocore.exceptions.ClientError as exc:
        print(f"skip {name}: {exc.response['Error']['Message']}")

---
## 9. Appendix: failures worth recognising

### 9.1 `pe_dim must be 64 for fp8_ds_mla`

Seen on 96 GiB cards (`ml.g7e.48xlarge`) at util 0.90 and 131072 context. The endpoint
provisions, then crash-loops until the health check expires.

```
RuntimeError: concat_and_cache_mla, csrc/libtorch_stable/cache_kernels.cu:866,
              pe_dim must be 64 for fp8_ds_mla
```

vLLM's FP8 DeepSeek-MLA KV-cache write kernel asserts the positional-encoding dimension is 64.
GLM-5.3-Flash sets `qk_rope_head_dim: 0` and `mla_use_nope: true`, so `pe_dim` is **0** and the
kernel has no NoPE path.

**This is not an architecture problem, it is memory pressure.** The preceding error:

```
OOM on device 0 while trying to allocate 20971520 bytes
  (free: 5701632, total: 101973819392)
```

`total` is 95 GiB with 5.7 MB free. The profiler exhausted its budget, vLLM fell back to an FP8
KV cache to fit, and that routed into `fp8_ds_mla`. On 141 GiB cards there was room, so it
stayed on the BF16 MLA path and booted. Expect this on any 96 GiB card at aggressive settings.
Mitigate with lower `--gpu-memory-utilization`, shorter `--max-model-len`, lower
`--max-num-seqs`, and an explicit `--kv-cache-dtype auto`.

A consequence worth internalising: a config "sized against the tightest pool member" is **not**
automatically safe across instance pools. A config that boots on 141 GiB cards can die on 96 GiB.

### 9.2 `no kernel image is available for execution on the device`

Not this container, but the same family of problem, and it hits people on new hardware:

```
UserWarning: NVIDIA RTX PRO 4500 Blackwell ... sm_120 is not compatible with the
current PyTorch installation. The current PyTorch install supports CUDA
capabilities sm_50 sm_60 sm_70 sm_75 sm_80 sm_86 sm_90.
```

Blackwell needs **PyTorch 2.7.0 or later built with CUDA 12.8 or newer**. A `-cu128` tag on an
image does not prove the bundled torch wheel has sm_120 cubins; the CUDA toolkit version and
the compiled arch list are different things. Settle it in one line instead of a 700 s
deploy-and-fail:

```bash
docker run --rm --entrypoint python3 <image> \
  -c "import torch; print(torch.__version__); print(torch.cuda.get_arch_list())"
```

Separately compiled extensions each have their own arch list, so `flash-attn`,
`bitsandbytes`, and `deepspeed` can fail the same way after torch itself is fine.

### 9.3 Capacity

`InsufficientInstanceCapacity` was the dominant obstacle, on multiple accounts and instance
types, and takes about 31 minutes to surface. **Quota and capacity are unrelated**; quota was
24 for every type involved.

Two mitigations:

- **Instance pools**, shown in section 3.4. Ordered fallback, up to 5 types. Gives you one
  endpoint on whichever type had capacity, so it cannot pin a type for comparison work.
- **Training plans**, via `CapacityReservationConfig` with an `MlReservationArn`. The plan must
  be created with target resource **`endpoint`**; a `training-job` plan will not attach. One
  plan per endpoint. A plan in `Scheduled` state is not yet usable.

Training plans are **p-family only**. `SearchTrainingPlanOfferings` rejects `ml.g7e.48xlarge`
as an invalid `instanceType`, so some types can only be retried, never reserved. A 24 h
watchdog firing roughly 35 attempts obtained zero g7e instances in `us-east-2`.

### 9.4 Local environment, if you run this from Windows

- `pip install sagemaker` fails with `WinError 206` (path too long) under Store Python. Install
  into a venv at a short path, e.g. `python -m venv %USERPROFILE%\bm`.
- The AWS CLI v1 has no `aws logs tail`. Use `filter-log-events` or boto3.
- The CLI emits a spurious `File association not found for extension .py` line to stderr that
  corrupts redirected JSON. Prefer boto3 over CLI-plus-parse.
- Non-ASCII output can raise `UnicodeEncodeError` on a cp1252 console. Keep operational output
  ASCII-only, or force `PYTHONIOENCODING=utf-8`. This is not cosmetic: see section 3.5.